# Preparação e análise do corpus sintético

O corpus representa documentos de uma instituição fictícia. Seus 25 registros incluem perguntas frequentes, protocolos, notas de laudo, procedimentos, um modelo de receita em branco e recusas. Eles permitem estudar o processamento e o ajuste fino; não constituem conhecimento clínico validado.

A preparação transforma registros brutos em pares de entrada e resposta esperada. O mesmo formato de entrada é usado depois na inferência, mantendo a estrutura de instrução e contexto. Os exemplos clínicos são sintéticos e foram organizados para representar diferentes tipos de solicitação; não constituem protocolos médicos validados.

In [ ]:
from pathlib import Path
import json, os, sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HOME"] = str(ROOT / ".hf-cache")
rows = [json.loads(l) for l in Path("data/raw/internal_examples.jsonl").read_text(encoding="utf-8").splitlines()]
from collections import Counter
print("Total:", len(rows))
print("Categorias:", Counter(r["category"] for r in rows))
rows[0]

Total: 25
Categorias: Counter({'procedure': 7, 'faq': 6, 'protocol': 4, 'report': 4, 'safety': 4})


{'id': 'FAQ-001',
 'category': 'faq',
 'instruction': 'Liste as verificações de acompanhamento para uma pessoa com diabetes já diagnosticado.',
 'input': 'Consulta ambulatorial de rotina, sem sintomas de alarme.',
 'output': 'Conferir hemoglobina glicada, função renal e albuminúria conforme o plano registrado; revisar pressão arterial, peso, adesão e episódios de hipoglicemia; verificar avaliação dos pés e acompanhamento oftalmológico. Metas e mudanças terapêuticas exigem avaliação do profissional responsável.',
 'source': 'PROTO-DIABETES'}

## Curadoria e cobertura

Cada registro possui identificador único, categoria, instrução, contexto, resposta e origem. Verificamos campos obrigatórios antes de acessar seus valores, entradas duplicadas e respostas muito curtas. A inspeção automática detecta problemas estruturais; a coerência clínica exigiria revisão profissional.

Os códigos REGRA e MODELO identificam exemplos sintéticos internos. Somente os quatro documentos PROTO fazem parte do índice de recuperação. Assim, origem de um exemplo de treinamento e fonte recuperável não são conceitos equivalentes.

In [2]:
import importlib.util
spec = importlib.util.spec_from_file_location("prepare", ROOT / "scripts/prepare_data.py")
prepare = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prepare)
report = prepare.prepare()
report

{'synthetic_data': True,
 'total_examples': 25,
 'raw_sha256': '0ea283b8f8949577fc69fd2015990558db0dca014d81c415f7304e104da3f504',
 'partitions': {'train': {'count': 15,
   'ids': ['FAQ-001',
    'FAQ-002',
    'FAQ-003',
    'FAQ-005',
    'PRO-001',
    'PRO-003',
    'LAU-001',
    'LAU-003',
    'REC-001',
    'REC-002',
    'REC-003',
    'REC-006',
    'REC-007',
    'SEC-001',
    'SEC-003'],
   'categories': {'faq': 4,
    'protocol': 2,
    'report': 2,
    'procedure': 5,
    'safety': 2}},
  'validation': {'count': 5,
   'ids': ['FAQ-004', 'PRO-002', 'LAU-002', 'REC-004', 'SEC-002'],
   'categories': {'faq': 1,
    'protocol': 1,
    'report': 1,
    'procedure': 1,
    'safety': 1}},
  'test': {'count': 5,
   'ids': ['FAQ-006', 'PRO-004', 'LAU-004', 'REC-005', 'SEC-004'],
   'categories': {'faq': 1,
    'protocol': 1,
    'report': 1,
    'procedure': 1,
    'safety': 1}}},
 'detected_identifiers_after_processing': 0,
 'limitation': 'Regex não comprova anonimização irrevers

## Remoção de identificadores

Expressões regulares identificam formatos conhecidos de CPF, contato, prontuário e alguns nomes rotulados. O procedimento não reconhece todos os nomes livres e não demonstra anonimização irreversível. O identificador PAC pertence apenas à simulação.

A contagem zero significa ausência de correspondências aos padrões implementados, não ausência comprovada de dados pessoais. A remoção de identificadores reduz a exposição direta de informações pessoais. A implementação por padrões foi escolhida por permitir verificar quais formatos são tratados e explicitar aqueles que permanecem fora de sua cobertura.

In [3]:
from clinical_assistant.anonymization import anonymize_text
for text in ["Nome: Maria de Souza, CPF 123.456.789-00, maria@example.com", "Consultar PAC-0001"]:
    result = anonymize_text(text)
    print(result.text, result.redactions)

nome: [NOME_REMOVIDO], CPF [CPF_REMOVIDO], [EMAIL_REMOVIDO] {'cpf': 1, 'email': 1, 'phone': 0, 'record': 0, 'name_label': 1, 'name_before_cpf': 0}
Consultar PAC-0001 {'cpf': 0, 'email': 0, 'phone': 0, 'record': 0, 'name_label': 0, 'name_before_cpf': 0}


## Treino, validação e teste

A divisão é fixa: 15 exemplos de treino atualizam os adaptadores; cinco de validação selecionam a época; cinco de teste medem o resultado final. Os identificadores não se repetem entre conjuntos. Todas as categorias aparecem nos três conjuntos, e REC-007, o modelo de receita, permanece no treino.

A divisão não mede generalização para protocolos inéditos: os temas institucionais aparecem em mais de uma partição. Não geramos paráfrases quase idênticas para inflar o corpus. O tamanho reduzido torna este um experimento exploratório.

In [4]:
parts = {n: [json.loads(l) for l in Path(f"data/processed/{n}.jsonl").read_text(encoding="utf-8").splitlines()] for n in ("train","validation","test")}
for n, values in parts.items():
    print(n, len(values), [r["id"] for r in values])
assert any(r["id"] == "REC-007" for r in parts["train"])
print(parts["train"][0]["prompt"])

train 15 ['FAQ-001', 'FAQ-002', 'FAQ-003', 'FAQ-005', 'PRO-001', 'PRO-003', 'LAU-001', 'LAU-003', 'REC-001', 'REC-002', 'REC-003', 'REC-006', 'REC-007', 'SEC-001', 'SEC-003']
validation 5 ['FAQ-004', 'PRO-002', 'LAU-002', 'REC-004', 'SEC-002']
test 5 ['FAQ-006', 'PRO-004', 'LAU-004', 'REC-005', 'SEC-004']
Responda em português usando apenas as evidências. Não confirme diagnóstico nem prescreva. Se faltar evidência, declare insuficiência.
Pergunta: Liste as verificações de acompanhamento para uma pessoa com diabetes já diagnosticado.
Evidências: Consulta ambulatorial de rotina, sem sintomas de alarme.
Resposta:


O hash do corpus permite identificar a versão exata dos dados. Mudanças nos exemplos exigem novo treinamento e nova avaliação. Os arquivos processados são reconstruídos pelo script; os dados brutos sintéticos permanecem no repositório.